In [55]:
# 解析军人图片
from pathlib import Path
import jsonlines
INPATH = "/Data_two/wyw/data/CETC_PRODUCT/merge/enwiki_task2.jsonl"
inpath = Path(INPATH)
outpath = "/Data_two/wyw/data/CETC_PRODUCT/task7/enwiki_for-download_20251126141200.jsonl"

In [56]:
need_domains = {"军事与安全", "政治与公共事务"}
def is_need(record):
    if set(record["domain_and_region_classifier"]["domains"]) & need_domains:
        return True
    else:
        return False

In [ ]:
domains = set()
need_records = []
with jsonlines.open(inpath) as reader, jsonlines.open(outpath, mode = "w") as writer:
    for r in reader:
        if is_need(r):
            writer.write(r)

In [60]:
# 准备元信息，先准备所有的page_id
extract_inpaths = sorted(Path("/Data_two/wyw/data/CETC_PRODUCT/task7").glob("*_for-download*"), reverse=True)

In [61]:
extract_inpaths

[PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task7/zhwiki_for-download_20251126141200.jsonl'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/task7/enwiki_for-download_20251126141200.jsonl')]

In [ ]:
import orjsonl
zhwiki_need_ids = set()
enwiki_need_ids = set()
for p in extract_inpaths:
    cur_wiki = p.stem.split("_", maxsplit=1)[0]
    with jsonlines.open(p) as reader:
        for json_obj in reader.iter(skip_invalid = True, skip_empty=True):
            if cur_wiki == "zhwiki":
                zhwiki_need_ids.add( json_obj["page_id"])
            elif cur_wiki == "enwiki":
                enwiki_need_ids.add( json_obj["page_id"])
    

In [70]:
len(zhwiki_need_ids), len(enwiki_need_ids)

(34046, 102858)

In [126]:
# 开始进行下一轮合并
raw_inpaths = sorted(Path("/Data_two/wyw/data/CETC_PRODUCT/raw/enwiki").glob("*.jsonl.zst"))

In [125]:
raw_inpaths

[PosixPath('/Data_two/wyw/data/CETC_PRODUCT/raw/zhwiki/0.jsonl.zst'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/raw/zhwiki/1.jsonl.zst'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/raw/zhwiki/2.jsonl.zst'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/raw/zhwiki/3.jsonl.zst'),
 PosixPath('/Data_two/wyw/data/CETC_PRODUCT/raw/zhwiki/4.jsonl.zst')]

In [127]:
zhwiki_raw_infos = dict()
enwiki_raw_infos = dict()

for p in raw_inpaths:
    cur_wiki = p.parent.stem
    if cur_wiki == "zhwiki":
        need_ids, raw_infos = zhwiki_need_ids, zhwiki_raw_infos
    elif cur_wiki == "enwiki":
        need_ids, raw_infos = enwiki_need_ids, enwiki_raw_infos
        
    for json_obj in orjsonl.stream(p):
        if json_obj["page_id"] in need_ids:
            raw_infos[json_obj["page_id"]] = json_obj


In [129]:
len(zhwiki_raw_infos), len(enwiki_raw_infos)

(0, 102858)

In [ ]:
# 进行元信息合并
zhwiki_raw_info

In [100]:
def parse_rawinfo(record):
    noneed_subcols = ["__template__", "__meta__"]
    try:
        infobox = {k: v for k, v in record["infoboxes_info"][0].items() if k not in noneed_subcols}
    except:
        infobox = None
    return {
        "page_id" : record["page_id"],
        "title" : record["title"],
        "abstract" : record["abstract"],
        "infobox" : infobox,
        "entity_classification" : record["entity_classification"]["result"],
        "domain_and_region_classifier" : record["domain_and_region_classifier"]["result"]
    }

In [131]:
outpath = "/Data_two/wyw/data/CETC_PRODUCT/task7/enwiki_for-picture_20251126152000.jsonl"
with jsonlines.open(outpath, mode = "w") as writer:
    for _id in enwiki_need_ids:
        wiki_info = parse_rawinfo(enwiki_raw_infos.get(_id))
        writer.write(wiki_info)

In [119]:
zhwiki_info

{'page_id': '7384351',
 'title': '侯賽因·哈卡尼',
 'abstract': '侯賽因·哈卡尼（，，），巴基斯坦记者、学者和政治活动家。他曾經担任巴基斯坦驻斯里兰卡的高級專員和驻美国大使，现在就职于美国哈德遜研究所。 哈卡尼曾經先后担任納瓦茲·謝里夫和贝娜齐尔·布托的政治顾问和发言人，直到1999年因批评军人总统佩尔韦兹·穆沙拉夫而流亡。2008年，他获扎尔达里委任为驻美大使，但部分人批评哈卡尼过分亲美的立场。 2011年10月，媒体报道哈卡尼在海神之矛行动一周后向美国參謀長聯席會議主席迈克尔·马伦写的備忘錄，希望获得奥巴马政府的支持阻止巴国军人掌權并支持文人政府取締军方。“‘Memogate’ scandal deepens as American accuser threatens to tell all” The Guardian [12-01-2012]事件曝光后在巴基斯坦引起广泛争议，甚至有声音指哈卡尼叛国，事件被称作「備忘錄门」（Memogate）。“巴基斯坦前驻美大使被禁止离境” BBC中文，2011年12月1日哈卡尼因此辭職，回国后接受巴基斯坦最高法院的调查。“巴基斯坦任命新駐美國大使” VOA [23-11-2011]',
 'infobox': {'name': ['侯賽因', '哈卡尼'],
  'image': ['Husain Haqqani, Ambassador of Pakistan to the United States (2008-11)',
   'Director, South and Central Asia, Hudson Institute (16528040198).jpg'],
  'image_size': '250px',
  'office': '第24任巴基斯坦駐美國大使',
  'term_start': '2008年4月13日',
  'term_end': '2011年11月12日',
  'predecessor': 'Mahmud Ali Durrani',
  'successor': ['雪莉', '雷曼'],
  'office1': '巴基斯坦駐斯里蘭卡高級專員',
  'term_start1': '1992年5月11日',
  'term_end1'

In [114]:
len(zhwiki_raw_info)

9

In [113]:
zhwiki_id

'8391105'